In [ ]:
import os
from os.path import dirname

root_dir = dirname(os.getcwd())
os.chdir(root_dir)

In [ ]:
import yaml
import pickle
import torch
import numpy as np

from src.nn import *
from src.utils import *
from sklearn.model_selection import StratifiedKFold

In [ ]:
random_seed = [0, 42, 1234, 1337]
num_folds = 5
cnn_state_dict_list = []
nn_state_dict_list = []

In [ ]:
for seed in random_seed:
    for fold in range(1, num_folds+1):
        cnn_path = f'results/cnn/seed_{seed}/fold_{fold}/best_model.pth'
        nn_path = f'results/nn/seed_{seed}/fold_{fold}/best_model.pth'
        cnn_state_dict_list.append(torch.load(cnn_path))
        nn_state_dict_list.append(torch.load(nn_path))

In [ ]:
merged_cnn = ConvolutionalNeuralNetwork()
merged_cnn_state_dict = merged_cnn.state_dict()

for key in merged_cnn_state_dict.keys():
    cnn_params = [state_dict[key] for state_dict in cnn_state_dict_list]
    merged_cnn_state_dict[key] = torch.mean(torch.stack(cnn_params), dim=0)

merged_cnn.load_state_dict(merged_cnn_state_dict)

<All keys matched successfully>

In [ ]:
merged_nn = NeuralNetwork()
merged_nn_state_dict = merged_nn.state_dict()

for key in merged_nn_state_dict.keys():
    nn_params = [state_dict[key] for state_dict in nn_state_dict_list]
    merged_nn_state_dict[key] = torch.mean(torch.stack(nn_params), dim=0)

merged_nn.load_state_dict(merged_nn_state_dict)

<All keys matched successfully>

In [ ]:
with open('data/vehicle_data_kaggle_train.pkl', 'rb') as f:
    vehicle_data = pickle.load(f)

In [ ]:
skf = StratifiedKFold(n_splits=num_folds, shuffle=True)

for i, (_, test_index) in enumerate(skf.split(vehicle_data['data'], vehicle_data['label'])):
    test_subset = {
        'data': vehicle_data['data'][test_index],
        'label': vehicle_data['label'][test_index],
    }

    test_dl = get_dataloader(test_subset)

    merged_cnn.eval()
    merged_nn.eval()

    cnn_eval_metrics = {}
    nn_eval_metrics = {}

    cnn_true_label = []
    cnn_pred_label = []

    nn_true_label = []
    nn_pred_label = []

    with torch.no_grad():
        for inputs, labels in test_dl:
            cnn_outputs = merged_cnn(inputs)
            nn_outputs = merged_nn(inputs)
            _, cnn_preds = torch.max(cnn_outputs, 1)
            _, nn_preds = torch.max(nn_outputs, 1)

            cnn_true_label.extend(labels.cpu().numpy())
            cnn_pred_label.extend(cnn_preds.cpu().numpy())

            nn_true_label.extend(labels.cpu().numpy())
            nn_pred_label.extend(nn_preds.cpu().numpy())

        cnn_eval_metrics = calculate_metrics(cnn_true_label, cnn_pred_label)
        nn_eval_metrics = calculate_metrics(nn_true_label, nn_pred_label)
    print(f'Fold {i+1} CNN Evaluation Metrics: {cnn_eval_metrics}')
    print(f'Fold {i+1} NN Evaluation Metrics: {nn_eval_metrics}')
    print('-'*80)

Fold 1 CNN Evaluation Metrics: {'accuracy': 0.9161290322580645, 'precision': 0.9161290322580645, 'recall': 0.9161290322580645, 'f1_score': 0.9161290322580645}
Fold 1 NN Evaluation Metrics: {'accuracy': 0.9225806451612903, 'precision': 0.9225806451612903, 'recall': 0.9225806451612903, 'f1_score': 0.9225806451612903}
--------------------------------------------------------------------------------
Fold 2 CNN Evaluation Metrics: {'accuracy': 0.9032258064516129, 'precision': 0.9032258064516129, 'recall': 0.9032258064516129, 'f1_score': 0.9032258064516129}
Fold 2 NN Evaluation Metrics: {'accuracy': 0.9225806451612903, 'precision': 0.9225806451612903, 'recall': 0.9225806451612903, 'f1_score': 0.9225806451612903}
--------------------------------------------------------------------------------
Fold 3 CNN Evaluation Metrics: {'accuracy': 0.9096774193548387, 'precision': 0.9096774193548387, 'recall': 0.9096774193548387, 'f1_score': 0.9096774193548387}
Fold 3 NN Evaluation Metrics: {'accuracy': 0.

In [ ]:
cnn_dir = 'results/cnn/averaged_model'
nn_dir = 'results/nn/averaged_model'
create_dir(cnn_dir)
create_dir(nn_dir)

In [ ]:
torch.save(merged_cnn.state_dict(), f'{cnn_dir}/best_model.pth')
torch.save(merged_nn.state_dict(), f'{nn_dir}/best_model.pth')